# Tutorial 02 — Code Agent（會寫程式 + 執行）
## NCHC LLM 工作坊 2026 · NVIDIA NeMo Agent Toolkit

---

### 你會學到什麼

- **ReAct**（Reasoning + Acting）agent pattern
- 用內建的 `code_execution` tool 讓 agent **自己寫 Python 並執行**
- 為什麼 ReAct 比 `tool_calling_agent` 更適合「邊跑邊修」這類任務
- 怎麼用 `--override` 切換 verbose、temperature 等行為

### 為什麼用 code_execution，不用搜尋類 tool？

`code_execution` 比 `wiki_search` 更能凸顯 ReAct：
- 算錯了 → 看到 `Traceback` → 自己修 bug → 再跑一次
- 想驗證結果 → 再寫一段檢查的 code
- 完全離線、不依賴第三方網站、結果可重現

### `code_execution` 的運作

Sandbox 是 `nvidia-nat` 內附的 Flask server（`local_sandbox_server.py`）。
我們不需要 docker —— Step 1 會把它以 Python 子程序的形式拉起來在 `http://127.0.0.1:6000`。

```
LLM 寫 Python ──► POST /execute ──► sandbox exec() ──► stdout/stderr ──► LLM 看 Observation
```

### ReAct 迴圈在這個情境下長怎樣

```
問題
   │
   ▼
┌──────────────────────────────────────────────────────────┐
│                      react_agent                         │
│                                                          │
│  Thought: "先用 X 公式算一次"                            │
│       ▼                                                  │
│  Action: py                                              │
│  Action Input: {"generated_code": "..."}                 │
│       ▼                                                  │
│  Observation: stdout / stderr                            │
│       ▼                                                  │
│  Thought: "結果不對，我看到 ZeroDivisionError，改用 ..." │
│       ▼                                                  │
│  Action: py（換新程式）                                  │
│       ▼                                                  │
│  Final Answer: ...                                       │
└──────────────────────────────────────────────────────────┘
```

`react_agent` 因為每一步都能看到 Observation，**自然支援多 round 程式跑修**。

1. **多步 pipeline**：第一段 code 的輸出當第二段的輸入
2. **解 + 驗證**：先解、再寫一段獨立的 code 來驗證答案
3. **迭代求解**：寫 loop、看是否收斂、回報所需迭代次數

> **關鍵：** 第二次寫的 code 必須等第一次的 Observation 才知道要怎麼改。
> 這就是 ReAct 比 `tool_calling_agent` 強的地方。

> **開始前：** 從工作坊根目錄執行 `uv sync` 與 `uv run jupyter lab`。

---
## 步驟 1 — 確認環境 + 啟動 sandbox

下面這個 cell 會：
1. 確認 `nat` 已安裝
2. 確認 `NVIDIA_API_KEY` 已設定（沒有就跳出輸入框）
3. 在背景啟動 `local_sandbox_server`（如已在跑就直接重用），讓 `code_execution` tool 可以打 `http://127.0.0.1:6000`


In [ ]:
import os, sys, subprocess, time, getpass
import urllib.request
from pathlib import Path

# (1) nat installed?
r = subprocess.run(["nat", "--version"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

# (2) API key
if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    key = getpass.getpass("NVIDIA_API_KEY 尚未設定。請輸入你的 key（nvapi-...）：")
    os.environ["NVIDIA_API_KEY"] = key.strip()
print("NVIDIA_API_KEY ✓")

os.environ["NAT_CONFIG_DIR"] = os.path.abspath("nat_config")
print("NAT_CONFIG_DIR ✓")

# (3) Sandbox server
SANDBOX_URL = "http://127.0.0.1:6000/"

def _sandbox_alive():
    try:
        urllib.request.urlopen(SANDBOX_URL, timeout=1)
        return True
    except Exception:
        return False

if _sandbox_alive():
    print("sandbox already running ✓")
else:
    import nat.tool.code_execution.local_sandbox as _ls
    server_py = Path(_ls.__file__).parent / "local_sandbox_server.py"
    log_f = open("/tmp/sandbox.log", "a", buffering=1)
    subprocess.Popen(
        [sys.executable, str(server_py)],
        stdout=log_f, stderr=log_f,
        start_new_session=True,
    )
    for _ in range(20):
        if _sandbox_alive():
            print("sandbox started ✓ (port 6000)")
            break
        time.sleep(0.5)
    else:
        print("⚠ sandbox 沒起來，查 /tmp/sandbox.log")

---
## 步驟 2 — 看 Config

與 Tutorial 01 兩個關鍵差別：
1. 主要 tool 改成 `code_execution`——agent 寫的 Python 會被丟到本地 sandbox 執行
2. `react_agent` 採用顯式的 **Thought → Action → Observation** 迴圈


---
## 步驟 3 — 執行 Code Agent

讓 agent 寫一段算式 → sandbox 跑 → agent 看結果回答。
注意 verbose 中 **Thought / Action / Observation** 三段都會出現。


In [ ]:
!nat run \
    --config_file tutorial_02_code_agent/configs/config.yml \
    --input "計算 2 ** 100 並告訴我答案有幾位數，並用中文回應"

### 解讀輸出

```
Thought: I'll compute the value and the digit count in Python.
Action: py
Action Input: {"generated_code": "v=2**100\nprint(v); print(len(str(v)))"}
Observation: 1267650600228229401496703205376
             31
Thought: I have the answer.
Final Answer: 2^100 = 1267650600228229401496703205376, which has 31 digits.
```

三件事值得學員觀察：
- **Action Input 是 JSON**（`{"generated_code": "..."}`），不是純文字
- **Observation 是 sandbox 的 stdout**（多行也照印）
- LLM 是看了 Observation 才能寫出 Final Answer
- 第一次寫的 code 若沒 `print()` 通常會看到第二輪 Action 補印——天然的 ReAct 自我修正


---
## 步驟 4 — 比較 Agent 類型

| Agent 類型 | 推理風格 | 適用 |
|---|---|---|
| `tool_calling_agent` | LLM 結構化的 tool-calling API | 一次決定要呼叫什麼、快速 |
| `react_agent` | 顯式 Thought / Action / Observation 迴圈 | 多步推理、寫 code + 驗證 |

想看差異：打開 `tutorial_02_code_agent/configs/config.yml`，把 `workflow._type` 改成
`tool_calling_agent`，存檔後重跑上面任一個 cell。比較中間步驟長怎樣。


---
## 總結

| 概念 | 學到了什麼 |
|---|---|
| **`code_execution`** | 把 LLM 寫的 Python 丟到 sandbox 跑、回傳 stdout / stderr |
| **本地 sandbox server** | 內附的 Flask 程序（不必 docker），由 notebook setup cell 拉起來 |
| **ReAct + code** | 每一段 code 都是一個 Action；下一段是看了 Observation 才寫的 |
| **多步 pipeline** | 中間結果驅動下一步程式 |
| **解 + 驗證** | 一段算、第二段檢查；最能凸顯 ReAct 的價值 |
| **`--override`** | CLI 級別調整 verbose / temperature 等欄位 |

➡️ **下一個：** `tutorial_03_custom_tool.ipynb`

---
## 步驟 5 — 關閉 Sandbox

Step 1 啟動的 `local_sandbox_server` 是 jupyter kernel 的子程序，
**kernel 結束後也會跟著消失**；但若你想提早把 6000 port 釋放，跑下面這個 cell。

之後想再用 code agent，重跑 Step 1 即可（會自動重新啟動）。


In [ ]:
import subprocess, urllib.request

def _alive():
    try:
        urllib.request.urlopen("http://127.0.0.1:6000/", timeout=1)
        return True
    except Exception:
        return False

if _alive():
    subprocess.run(["pkill", "-f", "local_sandbox_server.py"], check=False)
    print("sandbox 已關閉 ✓")
else:
    print("sandbox 不在執行中，略過")